# Notebook 07 -- Feature Engineering: Tabular

<div style="border-left: 4px solid #4680a7; padding: 10px 15px; margin: 10px 0; background: #4e6681;">

**Objective:** Engineer predictive features from the cleaned tabular data by applying domain-driven transformations, statistical aggregations, interaction construction, and encoding strategies. Produce a versioned tabular feature matrix aligned with findings from the tabular EDA (notebook 04).

**Answers:** *"What structured features, derived from raw tabular columns, maximize predictive signal for adoption speed?"*

</div>

| Item | Detail |
|------|--------|
| **Dependencies** | `data/cleaned/train.parquet`, `data/cleaned/test.parquet`, reference tables, `reports/eda_tabular_findings.json` |
| **Artifacts** | `data/features/tabular/v1/train.parquet`, `data/features/tabular/v1/test.parquet`, `data/features/tabular/v1/schema.json`, `reports/feature_engineering_tabular.json` |
| **Scope** | Tabular features only -- no text, image, or metadata features |
| **Runtime** | < 1 min |

---
## 1. Imports & Configuration

In [1]:
# -- Standard library & third-party imports ------------------------------------
from __future__ import annotations

import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# -- Project imports -----------------------------------------------------------
from adoption_accelerator import config as cfg
from adoption_accelerator.data.ingestion import load_parquet, save_parquet
from adoption_accelerator.features.tabular import (
    FEATURE_COLUMNS,
    FEATURE_DESCRIPTIONS,
    encode_pet_type,
    create_health_care_score,
    recode_care_features,
    transform_numeric_features,
    create_name_features,
    create_breed_features,
    create_color_features,
    create_interaction_features,
    compute_rescuer_aggregates,
    apply_rescuer_aggregates,
    frequency_encode,
    engineer_tabular_features,
)
from adoption_accelerator.features.registry import (
    build_column_descriptors,
    compute_config_hash,
    save_feature_schema,
)
from adoption_accelerator.utils.logging import setup_logging

setup_logging()

# -- Reproducibility & display ------------------------------------------------
SEED = cfg.SEED
np.random.seed(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# -- Output paths --------------------------------------------------------------
FEATURES_DIR = cfg.DATA_FEATURES / "tabular" / "v1"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = cfg.REPORTS_DIR / "feature_engineering_tabular.json"

print("Imports and configuration loaded.")
print(f"   Seed          : {SEED}")
print(f"   Features dir  : {FEATURES_DIR}")
print(f"   Report path   : {REPORT_PATH}")

Imports and configuration loaded.
   Seed          : 42
   Features dir  : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1
   Report path   : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_engineering_tabular.json


---
## 2. Load Cleaned Data & EDA Findings

In [2]:
# -- Load cleaned train and test DataFrames ------------------------------------
train = load_parquet(cfg.DATA_CLEANED / "train.parquet")
test = load_parquet(cfg.DATA_CLEANED / "test.parquet")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nTrain columns:\n  {list(train.columns)}")
train.head(3)

20:42:59  INFO      Loaded Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned\train.parquet (14993 rows, 25 cols)
20:42:59  INFO      Loaded Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned\test.parquet (3972 rows, 24 cols)


Train shape : (14993, 25)
Test shape  : (3972, 24)

Train columns:
  ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', 'has_name']


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,FurLength,Vaccinated,Dewormed,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed,has_name
0,2,Nibble,3,299,0,1,1,7,0,1,1,2,2,2,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0000,2,1
1,2,NaN,1,265,0,1,1,2,0,2,2,3,3,3,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0000,0,0
2,1,Brisco,1,307,0,1,2,7,0,2,2,1,1,2,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0000,3,1


In [3]:
# -- Load EDA tabular findings to inform feature strategy ----------------------
eda_path = cfg.REPORTS_DIR / "eda_tabular_findings.json"
with open(eda_path, "r", encoding="utf-8") as f:
    eda_findings = json.load(f)

print("EDA findings loaded.")
print(f"   Target imbalance ratio : {eda_findings['imbalance_ratio']}")
print(f"   Mixed breed %          : {eda_findings['breed_analysis']['mixed_breed_pct']}")
print(f"   Free listings %        : {eda_findings['fee_analysis']['free_pct']}")
print(f"   Unique rescuers        : {eda_findings['rescuer_analysis']['n_unique_rescuers']}")
print(f"\n   Interaction hypotheses:")
for h in eda_findings["interaction_hypotheses"]:
    print(f"     - {h}")

EDA findings loaded.
   Target imbalance ratio : 10.24
   Mixed breed %          : 28.19
   Free listings %        : 84.46
   Unique rescuers        : 5595

   Interaction hypotheses:
     - Age × Type — age distributions differ between dogs and cats across speed classes
     - Fee × State — fee impact on adoption speed varies by geographic region
     - Vaccinated × Health — vaccination interacts with health condition in speed prediction


In [4]:
# -- Validation Gate G07-1: Shape & columns ------------------------------------
EXPECTED_TRAIN_ROWS = 14_993
EXPECTED_TEST_ROWS = 3_972

assert train.shape[0] == EXPECTED_TRAIN_ROWS, (
    f"G07-1 FAIL: Expected {EXPECTED_TRAIN_ROWS} train rows, got {train.shape[0]}"
)
assert test.shape[0] == EXPECTED_TEST_ROWS, (
    f"G07-1 FAIL: Expected {EXPECTED_TEST_ROWS} test rows, got {test.shape[0]}"
)
assert "AdoptionSpeed" in train.columns, "G07-1 FAIL: AdoptionSpeed missing from train"
assert "AdoptionSpeed" not in test.columns, "G07-1 FAIL: AdoptionSpeed present in test"
assert "PetID" in train.columns and "PetID" in test.columns, "G07-1 FAIL: PetID missing"

print("G07-1 PASS: Cleaned data loaded with expected shapes.")
print(f"   Train: {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"   Test : {test.shape[0]:,} rows x {test.shape[1]} cols")

G07-1 PASS: Cleaned data loaded with expected shapes.
   Train: 14,993 rows x 25 cols
   Test : 3,972 rows x 24 cols


---
## 3. Feature Engineering Pipeline

<div style="border-left: 4px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #5e5535;">

**Strategy:** All transformations are applied via `src/adoption_accelerator/features/tabular.py` functions. Statistics that require training data (frequency maps, rescuer aggregates) are fitted on train and applied to both splits. This section walks through each transformation group for transparency, then runs the full orchestrator.

</div>

### 3.1 Transformation Preview (Step-by-Step)

Before running the full pipeline, we preview each transformation group on a sample to verify correctness and understand the output.

In [5]:
# -- Step 3: Pet type binary encoding ------------------------------------------
sample = train.head(5).copy()
sample = encode_pet_type(sample)
print("encode_pet_type:")
print(sample[["PetID", "Type", "is_dog"]].to_string(index=False))

encode_pet_type:
    PetID  Type  is_dog
86e1089a3     2       0
6296e909a     2       0
3422e4906     1       1
5842f1ff5     1       1
850a43f90     1       1


In [6]:
# -- Step 5-6: Health-care composite score & care recoding ---------------------
sample = train.head(5).copy()
sample = create_health_care_score(sample)
print("create_health_care_score (before recoding):")
print(sample[["PetID", "Vaccinated", "Dewormed", "Sterilized", "health_care_score"]].to_string(index=False))

sample = recode_care_features(sample)
print("\nrecode_care_features (after recoding):")
print(sample[["PetID", "Vaccinated", "Dewormed", "Sterilized", "health_care_score"]].to_string(index=False))

create_health_care_score (before recoding):
    PetID  Vaccinated  Dewormed  Sterilized  health_care_score
86e1089a3           2         2           2                  0
6296e909a           3         3           3                  0
3422e4906           1         1           2                  2
5842f1ff5           1         1           2                  2
850a43f90           2         2           2                  0

recode_care_features (after recoding):
    PetID  Vaccinated  Dewormed  Sterilized  health_care_score
86e1089a3           0         0           0                  0
6296e909a          -1        -1          -1                  0
3422e4906           1         1           0                  2
5842f1ff5           1         1           0                  2
850a43f90           0         0           0                  0


In [7]:
# -- Steps 7-10: Numeric transformations ---------------------------------------
sample = train.head(5).copy()
sample = transform_numeric_features(sample)
numeric_cols = [
    "Age", "log_age", "age_bin",
    "Fee", "log_fee", "is_free", "fee_per_pet",
    "Quantity", "log_quantity", "is_single_pet",
    "PhotoAmt", "log_photo_amt", "has_photos",
    "VideoAmt", "has_video",
]
print("transform_numeric_features:")
display(sample[["PetID"] + numeric_cols])

transform_numeric_features:


,PetID,Age,log_age,age_bin,Fee,log_fee,is_free,fee_per_pet,Quantity,log_quantity,is_single_pet,PhotoAmt,log_photo_amt,has_photos,VideoAmt,has_video
0,86e1089a3,3,1.3863,1,100,4.6151,0,100.0000,1,0.6931,1,1.0000,0.6931,1,0,0
1,6296e909a,1,0.6931,0,0,0.0000,1,0.0000,1,0.6931,1,2.0000,1.0986,1,0,0
2,3422e4906,1,0.6931,0,0,0.0000,1,0.0000,1,0.6931,1,7.0000,2.0794,1,0,0
3,5842f1ff5,4,1.6094,2,150,5.0173,0,150.0000,1,0.6931,1,8.0000,2.1972,1,0,0
4,850a43f90,1,0.6931,0,0,0.0000,1,0.0000,1,0.6931,1,3.0000,1.3863,1,0,0


In [8]:
# -- Step 11: Name features ----------------------------------------------------
sample = train.head(10).copy()
sample = create_name_features(sample)
print("create_name_features:")
print(sample[["PetID", "Name", "has_name", "name_length", "name_word_count"]].to_string(index=False))

create_name_features:
    PetID                    Name  has_name  name_length  name_word_count
86e1089a3                  Nibble         1            6                1
6296e909a                     NaN         0            0                0
3422e4906                  Brisco         1            6                1
5842f1ff5                    Miko         1            4                1
850a43f90                  Hunter         1            6                1
d24c30b4b                     NaN         0            0                0
1caa6fcdb                   BULAT         1            5                1
97aa9eeac Siu Pak & Her 6 Puppies         1           23                6
c06d167ca                     NaN         0            0                0
7a0942d61                   Kitty         1            5                1


In [9]:
# -- Step 12: Breed features ---------------------------------------------------
breed_freq_map = train["Breed1"].value_counts(normalize=True)
sample = train.head(10).copy()
sample = create_breed_features(sample, breed_freq_map)
print("create_breed_features:")
print(sample[["PetID", "Breed1", "Breed2", "is_mixed_breed", "breed_count", "breed1_frequency"]].to_string(index=False))
print(f"\nBreed1 frequency map: {len(breed_freq_map)} unique breeds")

create_breed_features:
    PetID  Breed1  Breed2  is_mixed_breed  breed_count  breed1_frequency
86e1089a3     299       0               0            1            0.0228
6296e909a     265       0               0            1            0.0839
3422e4906     307       0               0            1            0.3955
5842f1ff5     307       0               0            1            0.3955
850a43f90     307       0               0            1            0.3955
d24c30b4b     266       0               0            1            0.2424
1caa6fcdb     264     264               1            2            0.0197
97aa9eeac     307       0               0            1            0.3955
c06d167ca     265       0               0            1            0.0839
7a0942d61     265       0               0            1            0.0839

Breed1 frequency map: 175 unique breeds


In [10]:
# -- Steps 13-14: Color features & missing indicators -------------------------
sample = train.head(10).copy()
sample = create_color_features(sample)
print("create_color_features:")
print(sample[["PetID", "Color1", "Color2", "Color3", "color_count",
              "breed2_missing", "color2_missing", "color3_missing"]].to_string(index=False))

create_color_features:
    PetID  Color1  Color2  Color3  color_count  breed2_missing  color2_missing  color3_missing
86e1089a3       1       7       0            2               1               0               1
6296e909a       1       2       0            2               1               0               1
3422e4906       2       7       0            2               1               0               1
5842f1ff5       1       2       0            2               1               0               1
850a43f90       1       0       0            1               1               1               1
d24c30b4b       5       6       0            2               1               0               1
1caa6fcdb       1       0       0            1               0               1               1
97aa9eeac       1       2       7            3               1               0               0
c06d167ca       6       0       0            1               1               1               1
7a0942d61       1       7  

In [11]:
# -- Step 15: Interaction features ---------------------------------------------
sample = train.head(5).copy()
sample = encode_pet_type(sample)
sample = recode_care_features(sample)
sample = create_interaction_features(sample)
print("create_interaction_features:")
print(sample[["PetID", "Age", "is_dog", "age_x_type",
              "Health", "Vaccinated", "health_x_vaccinated"]].to_string(index=False))

create_interaction_features:
    PetID  Age  is_dog  age_x_type  Health  Vaccinated  health_x_vaccinated
86e1089a3    3       0      0.0000       1           0               0.0000
6296e909a    1       0      0.0000       1          -1              -1.0000
3422e4906    1       1      1.0000       1           1               1.0000
5842f1ff5    4       1      4.0000       1           1               1.0000
850a43f90    1       1      1.0000       1           0               0.0000


In [12]:
# -- Step 16: Rescuer aggregation features -------------------------------------
rescuer_stats = compute_rescuer_aggregates(train)
print(f"Rescuer aggregation: {len(rescuer_stats)} unique rescuers")
print(f"\nTop 5 rescuers by pet count:")
display(rescuer_stats.nlargest(5, "rescuer_pet_count"))

# Default values for cold-start rescuers
rescuer_defaults = {
    "rescuer_pet_count": float(rescuer_stats["rescuer_pet_count"].mean()),
    "rescuer_mean_photo_amt": float(rescuer_stats["rescuer_mean_photo_amt"].mean()),
    "rescuer_mean_fee": float(rescuer_stats["rescuer_mean_fee"].mean()),
}
print(f"\nCold-start defaults: {rescuer_defaults}")

Rescuer aggregation: 5595 unique rescuers

Top 5 rescuers by pet count:


,rescuer_pet_count,rescuer_mean_photo_amt,rescuer_mean_fee
RescuerID,,,
fa90fa5b1ee11c86938398b60abc32cb,459,3.5730,0.9804
aa66486163b6cbc25ea62a34b11c9b91,315,5.4984,6.8254
c00756f2bdd8fa88fc9f07a8309f7d5d,231,4.6753,19.3074
b53c34474d9e24574bcec6a3d3306a0d,228,3.1404,9.8684
ee2747ce26468ec44c7194e7d1d9dad9,156,4.7115,0.0000



Cold-start defaults: {'rescuer_pet_count': 2.6797140303842717, 'rescuer_mean_photo_amt': 3.476240634918213, 'rescuer_mean_fee': 23.9622745513916}


In [13]:
# -- Step 17: State frequency encoding -----------------------------------------
state_freq_map = train["State"].value_counts(normalize=True)
print("State frequency map (top 5):")
print(state_freq_map.head().to_string())
print(f"\n{len(state_freq_map)} unique states in training data")

State frequency map (top 5):
State
41326   0.5812
41401   0.2565
41327   0.0562
41336   0.0338
41330   0.0280

14 unique states in training data


---
## 4. Run Full Pipeline

<div style="border-left: 4px solid #27ae60; padding: 8px 12px; margin: 8px 0; background: #2d4a37;">

The orchestrator `engineer_tabular_features()` runs all transformation steps in sequence. Statistics (frequency maps, rescuer aggregates) are fitted on **train** and applied to both splits.

</div>

In [14]:
# -- Run full pipeline on TRAIN ------------------------------------------------
train_features, fitted_maps, train_log = engineer_tabular_features(
    train, split="train"
)
print(f"Train features: {train_features.shape}")
print(f"Steps executed: {train_log['steps']}")
print(f"\nNull counts (should all be 0):")
nulls = train_features.isna().sum()
non_zero_nulls = nulls[nulls > 0]
if len(non_zero_nulls) == 0:
    print("   No null values -- all features clean.")
else:
    print(non_zero_nulls.to_string())

20:43:42  INFO      Starting tabular feature engineering for split=train
20:43:42  INFO      Tabular FE complete for split=train: 14993 rows x 46 features


Train features: (14993, 46)
Steps executed: ['encode_pet_type', 'create_health_care_score', 'recode_care_features', 'transform_numeric_features', 'create_name_features', 'create_breed_features', 'create_color_features', 'create_interaction_features', 'rescuer_aggregates', 'state_frequency_encode']

Null counts (should all be 0):
   No null values -- all features clean.


In [15]:
# -- Run full pipeline on TEST (using train-fitted maps) -----------------------
test_features, _, test_log = engineer_tabular_features(
    test, split="test", fitted_maps=fitted_maps
)
print(f"Test features: {test_features.shape}")
print(f"Steps executed: {test_log['steps']}")
print(f"\nNull counts (should all be 0):")
nulls = test_features.isna().sum()
non_zero_nulls = nulls[nulls > 0]
if len(non_zero_nulls) == 0:
    print("   No null values -- all features clean.")
else:
    print(non_zero_nulls.to_string())

20:43:44  INFO      Starting tabular feature engineering for split=test
20:43:44  INFO      Tabular FE complete for split=test: 3972 rows x 46 features


Test features: (3972, 46)
Steps executed: ['encode_pet_type', 'create_health_care_score', 'recode_care_features', 'transform_numeric_features', 'create_name_features', 'create_breed_features', 'create_color_features', 'create_interaction_features', 'rescuer_aggregates', 'state_frequency_encode']

Null counts (should all be 0):
   No null values -- all features clean.


---
## 5. Feature Matrix Inspection

In [16]:
# -- Train feature matrix overview ---------------------------------------------
print(f"Train feature matrix: {train_features.shape[0]:,} rows x {train_features.shape[1]} columns")
print(f"\nColumn list:")
for i, col in enumerate(train_features.columns, 1):
    dtype = train_features[col].dtype
    print(f"  {i:2d}. {col:<30s}  {str(dtype):<10s}")

Train feature matrix: 14,993 rows x 46 columns

Column list:
   1. PetID                           str       
   2. is_dog                          int8      
   3. Gender                          int64     
   4. MaturitySize                    int64     
   5. FurLength                       int64     
   6. Health                          int64     
   7. Vaccinated                      int8      
   8. Dewormed                        int8      
   9. Sterilized                      int8      
  10. health_care_score               int8      
  11. Age                             int64     
  12. log_age                         float32   
  13. age_bin                         int8      
  14. Fee                             int64     
  15. log_fee                         float32   
  16. is_free                         int8      
  17. fee_per_pet                     float32   
  18. Quantity                        int64     
  19. log_quantity                    float32   
  20. is

In [17]:
# -- Data types summary --------------------------------------------------------
print("Data type counts:")
print(train_features.dtypes.value_counts().to_string())
print(f"\nMemory usage: {train_features.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")

Data type counts:
int8       17
int64      14
float32    11
int16       2
str         1
float64     1

Memory usage: 2.89 MB


In [19]:
# -- Descriptive statistics for numeric features -------------------------------
numeric_features = train_features.select_dtypes(include="number")
display(
    numeric_features.describe()
    .T.style.format("{:.4f}")
    .set_caption("Train Feature Matrix -- Descriptive Statistics")
)

,count,mean,std,min,25%,50%,75%,max
is_dog,14993.0000,0.5424,0.4982,0.0000,0.0000,1.0000,1.0000,1.0000
Gender,14993.0000,1.7762,0.6816,1.0000,1.0000,2.0000,2.0000,3.0000
MaturitySize,14993.0000,1.8620,0.5480,1.0000,2.0000,2.0000,2.0000,4.0000
FurLength,14993.0000,1.4675,0.5991,1.0000,1.0000,1.0000,2.0000,3.0000
Health,14993.0000,1.0366,0.1995,1.0000,1.0000,1.0000,1.0000,3.0000
Vaccinated,14993.0000,0.2688,0.6676,-1.0000,0.0000,0.0000,1.0000,1.0000
Dewormed,14993.0000,0.4413,0.6958,-1.0000,0.0000,1.0000,1.0000,1.0000
Sterilized,14993.0000,0.0858,0.5662,-1.0000,0.0000,0.0000,0.0000,1.0000
health_care_score,14993.0000,1.1603,1.1237,0.0000,0.0000,1.0000,2.0000,3.0000
Age,14993.0000,10.4521,18.1558,0.0000,2.0000,3.0000,12.0000,255.0000


In [20]:
# -- Train vs Test column alignment check --------------------------------------
train_cols = set(train_features.columns)
test_cols = set(test_features.columns)

only_train = train_cols - test_cols
only_test = test_cols - train_cols

print("Column alignment check:")
if not only_train and not only_test:
    print("   PASS: Train and test have identical column sets.")
else:
    if only_train:
        print(f"   Only in train: {only_train}")
    if only_test:
        print(f"   Only in test : {only_test}")

print(f"\nTrain: {train_features.shape} | Test: {test_features.shape}")

Column alignment check:
   PASS: Train and test have identical column sets.

Train: (14993, 46) | Test: (3972, 46)


---
## 6. Persist Feature Matrices & Schema

<div style="border-left: 4px solid #8e44ad; padding: 8px 12px; margin: 8px 0; background: #4a3554;">

Feature matrices are saved to `data/features/tabular/v1/` as versioned Parquet files. A `schema.json` records column metadata, row counts, and the config hash for reproducibility tracking.

</div>

In [21]:
# -- Save feature matrices to Parquet ------------------------------------------
train_path = save_parquet(train_features, FEATURES_DIR / "train.parquet")
test_path = save_parquet(test_features, FEATURES_DIR / "test.parquet")

print(f"Train features saved: {train_path}")
print(f"Test features saved : {test_path}")

20:44:16  INFO      Saved Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1\train.parquet (14993 rows, 0.46 MB)
20:44:16  INFO      Saved Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1\test.parquet (3972 rows, 0.13 MB)


Train features saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1\train.parquet
Test features saved : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1\test.parquet


In [22]:
# -- Generate and save schema.json ---------------------------------------------
feature_cols_no_id = [c for c in train_features.columns if c != "PetID"]

column_descriptors = build_column_descriptors(
    train_features[feature_cols_no_id],
    source="tabular",
    descriptions=FEATURE_DESCRIPTIONS,
)

config_for_hash = {
    "version": "v1",
    "modality": "tabular",
    "seed": SEED,
    "feature_columns": feature_cols_no_id,
}

schema_metadata = {
    "version": "v1",
    "modality": "tabular",
    "config_hash": compute_config_hash(config_for_hash),
    "n_rows_train": len(train_features),
    "n_rows_test": len(test_features),
    "n_features": len(feature_cols_no_id),
    "seed": SEED,
    "notes": "Tabular features engineered from cleaned data. "
             "Frequency maps and rescuer aggregates fitted on train only.",
}

schema_path = save_feature_schema(column_descriptors, schema_metadata, FEATURES_DIR / "schema.json")
print(f"Schema saved: {schema_path}")

20:44:19  INFO      Saved feature schema: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1\schema.json (45 columns, modality=tabular)


Schema saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\tabular\v1\schema.json


In [23]:
# -- Save engineering report ---------------------------------------------------
report = {
    "notebook": "07_feature_engineering_tabular",
    "train_shape": list(train_features.shape),
    "test_shape": list(test_features.shape),
    "n_features": len(feature_cols_no_id),
    "feature_columns": feature_cols_no_id,
    "train_log": {k: v for k, v in train_log.items() if k != "null_counts"},
    "test_log": {k: v for k, v in test_log.items() if k != "null_counts"},
    "rescuer_defaults": rescuer_defaults,
    "n_unique_rescuers_train": len(fitted_maps["rescuer_stats"]),
    "n_unique_breeds_train": len(fitted_maps["breed_freq_map"]),
    "n_unique_states_train": len(fitted_maps["state_freq_map"]),
    "config_hash": schema_metadata["config_hash"],
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"Engineering report saved: {REPORT_PATH}")

Engineering report saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_engineering_tabular.json


---
## 7. Validation Gate -- G07

In [24]:
#  VALIDATION GATE -- G07
# ======================================================================

gate_results = []

# G07-1: Cleaned train/test loaded with expected shape
try:
    assert train.shape[0] == EXPECTED_TRAIN_ROWS
    assert test.shape[0] == EXPECTED_TEST_ROWS
    gate_results.append(("G07-1", "PASS", "Cleaned data loaded with expected shapes"))
except AssertionError as e:
    gate_results.append(("G07-1", "FAIL", str(e)))

# G07-2: No target leakage
try:
    assert "AdoptionSpeed" not in train_features.columns, "AdoptionSpeed in train features"
    assert "AdoptionSpeed" not in test_features.columns, "AdoptionSpeed in test features"
    gate_results.append(("G07-2", "PASS", "No target leakage in feature matrices"))
except AssertionError as e:
    gate_results.append(("G07-2", "FAIL", str(e)))

# G07-3: No NaN values
try:
    train_nans = int(train_features.isna().sum().sum())
    test_nans = int(test_features.isna().sum().sum())
    assert train_nans == 0, f"Train has {train_nans} NaN values"
    assert test_nans == 0, f"Test has {test_nans} NaN values"
    gate_results.append(("G07-3", "PASS", "No NaN values in feature matrices"))
except AssertionError as e:
    gate_results.append(("G07-3", "FAIL", str(e)))

# G07-4: Train and test have identical column sets
try:
    assert set(train_features.columns) == set(test_features.columns), (
        f"Column mismatch: only_train={train_cols - test_cols}, only_test={test_cols - train_cols}"
    )
    gate_results.append(("G07-4", "PASS", "Train and test have identical column sets"))
except AssertionError as e:
    gate_results.append(("G07-4", "FAIL", str(e)))

# G07-5: Row counts preserved
try:
    assert len(train_features) == EXPECTED_TRAIN_ROWS, (
        f"Train rows: {len(train_features)} != {EXPECTED_TRAIN_ROWS}"
    )
    assert len(test_features) == EXPECTED_TEST_ROWS, (
        f"Test rows: {len(test_features)} != {EXPECTED_TEST_ROWS}"
    )
    gate_results.append(("G07-5", "PASS", f"Row counts preserved: {EXPECTED_TRAIN_ROWS:,} / {EXPECTED_TEST_ROWS:,}"))
except AssertionError as e:
    gate_results.append(("G07-5", "FAIL", str(e)))

# G07-6: PetID uniqueness
try:
    assert train_features["PetID"].is_unique, "Train PetID not unique"
    assert test_features["PetID"].is_unique, "Test PetID not unique"
    gate_results.append(("G07-6", "PASS", "PetID uniqueness maintained"))
except AssertionError as e:
    gate_results.append(("G07-6", "FAIL", str(e)))

# G07-7: Rescuer aggregates computed from train only
try:
    train_rescuers = set(train["RescuerID"].unique())
    stats_rescuers = set(fitted_maps["rescuer_stats"].index)
    assert stats_rescuers.issubset(train_rescuers), "Rescuer stats contain non-train rescuers"
    gate_results.append(("G07-7", "PASS", "Rescuer aggregates from train only"))
except AssertionError as e:
    gate_results.append(("G07-7", "FAIL", str(e)))

# G07-8: Ordinal features remain integer-typed
ordinal_cols = ["MaturitySize", "FurLength", "Health", "age_bin"]
ordinal_ok = True
ordinal_msgs = []
for col in ordinal_cols:
    if col in train_features.columns:
        dt = str(train_features[col].dtype)
        if "int" not in dt:
            ordinal_ok = False
            ordinal_msgs.append(f"{col}: {dt}")
if ordinal_ok:
    gate_results.append(("G07-8", "PASS", "All ordinal features are integer-typed"))
else:
    gate_results.append(("G07-8", "WARN", f"Non-integer ordinals: {ordinal_msgs}"))

# G07-9: Parquet files exist and are loadable
try:
    reload_train = pd.read_parquet(FEATURES_DIR / "train.parquet")
    reload_test = pd.read_parquet(FEATURES_DIR / "test.parquet")
    gate_results.append(("G07-9", "PASS", "Parquet files exist and are loadable"))
except Exception as e:
    gate_results.append(("G07-9", "FAIL", str(e)))

# G07-10: schema.json exists and is valid
try:
    with open(FEATURES_DIR / "schema.json", "r", encoding="utf-8") as f:
        loaded_schema = json.load(f)
    assert "columns" in loaded_schema, "schema.json missing 'columns' key"
    assert "version" in loaded_schema, "schema.json missing 'version' key"
    gate_results.append(("G07-10", "PASS", "schema.json exists and is valid JSON"))
except Exception as e:
    gate_results.append(("G07-10", "FAIL", str(e)))

# G07-11: Parquet round-trip integrity
try:
    assert reload_train.shape == train_features.shape, (
        f"Shape mismatch: {reload_train.shape} != {train_features.shape}"
    )
    assert reload_test.shape == test_features.shape, (
        f"Shape mismatch: {reload_test.shape} != {test_features.shape}"
    )
    assert list(reload_train.columns) == list(train_features.columns), "Column order mismatch"
    gate_results.append(("G07-11", "PASS", "Parquet round-trip integrity verified"))
except AssertionError as e:
    gate_results.append(("G07-11", "FAIL", str(e)))

# G07-12: Feature count matches schema
try:
    schema_n = loaded_schema["n_features"]
    actual_n = len(feature_cols_no_id)
    assert schema_n == actual_n, f"Schema says {schema_n}, actual is {actual_n}"
    gate_results.append(("G07-12", "PASS", f"Feature count matches schema: {actual_n}"))
except AssertionError as e:
    gate_results.append(("G07-12", "WARN", str(e)))

# -- Print gate summary -------------------------------------------------------
print("=" * 70)
print("  VALIDATION GATE -- G07 RESULTS")
print("=" * 70)
for gid, status, msg in gate_results:
    if status == "PASS":
        icon = "[PASS]"
    elif "WARN" in status:
        icon = "[WARN]"
    else:
        icon = "[FAIL]"
    print(f"  {icon} {gid}: {msg}")

passed = sum(1 for _, s, _ in gate_results if s == "PASS")
warned = sum(1 for _, s, _ in gate_results if "WARN" in s)
failed = sum(1 for _, s, _ in gate_results if "FAIL" in s)
total = len(gate_results)
print(f"\n  Result: {passed}/{total} passed, {warned} warnings, {failed} failures.")
print("=" * 70)

# Halt on critical failures
critical_ids = {"G07-1", "G07-2", "G07-3", "G07-4", "G07-5", "G07-6", "G07-7", "G07-9", "G07-10", "G07-11"}
critical_failures = [g for g in gate_results if g[1] == "FAIL" and g[0] in critical_ids]
assert len(critical_failures) == 0, f"Critical gate failures: {critical_failures}"

  VALIDATION GATE -- G07 RESULTS
  [PASS] G07-1: Cleaned data loaded with expected shapes
  [PASS] G07-2: No target leakage in feature matrices
  [PASS] G07-3: No NaN values in feature matrices
  [PASS] G07-4: Train and test have identical column sets
  [PASS] G07-5: Row counts preserved: 14,993 / 3,972
  [PASS] G07-6: PetID uniqueness maintained
  [PASS] G07-7: Rescuer aggregates from train only
  [PASS] G07-8: All ordinal features are integer-typed
  [PASS] G07-9: Parquet files exist and are loadable
  [PASS] G07-10: schema.json exists and is valid JSON
  [PASS] G07-11: Parquet round-trip integrity verified
  [PASS] G07-12: Feature count matches schema: 45

  Result: 12/12 passed, 0 warnings, 0 failures.


---
## 8. Feature Summary

In [25]:
# -- Feature catalog with descriptions ----------------------------------------
catalog_rows = []
for col in feature_cols_no_id:
    catalog_rows.append({
        "Feature": col,
        "Dtype": str(train_features[col].dtype),
        "Nulls": int(train_features[col].isna().sum()),
        "Description": FEATURE_DESCRIPTIONS.get(col, ""),
    })

catalog_df = pd.DataFrame(catalog_rows)
display(
    catalog_df.style
    .set_caption("Tabular Feature Catalog (v1)")
    .set_properties(**{"text-align": "left"})
    .hide(axis="index")
)

Feature,Dtype,Nulls,Description
is_dog,int8,0,"Binary: 1 = Dog, 0 = Cat"
Gender,int64,0,"Original gender code (1=Male, 2=Female, 3=Mixed)"
MaturitySize,int64,0,Ordinal maturity size (1=Small to 4=Extra Large)
FurLength,int64,0,"Ordinal fur length (1=Short, 2=Medium, 3=Long)"
Health,int64,0,Ordinal health condition (1=Healthy to 3=Serious Injury)
Vaccinated,int8,0,"Tristate recoded: 1=Yes, 0=No, -1=Not Sure"
Dewormed,int8,0,"Tristate recoded: 1=Yes, 0=No, -1=Not Sure"
Sterilized,int8,0,"Tristate recoded: 1=Yes, 0=No, -1=Not Sure"
health_care_score,int8,0,Count of Yes (=1) among Vaccinated/Dewormed/Sterilized (0-3)
Age,int64,0,Original age in months


---
## Summary

<div style="border-left: 4px solid #27ae60; padding: 10px 15px; margin: 10px 0; background: #232a3b;">

**Tabular Feature Engineering -- Complete**

1. **Binary encodings** -- `is_dog`, `is_free`, `is_single_pet`, `has_photos`, `has_video`, `is_mixed_breed`, missing indicators
2. **Ordinal preservation** -- `MaturitySize`, `FurLength`, `Health` kept as integers for tree-based models
3. **Health-care composite** -- `health_care_score` (0-3) reduces collinearity among Vaccinated/Dewormed/Sterilized (Cramer's V up to 0.748)
4. **Care recoding** -- Tristate {1, 0, -1} for cleaner ordinal interpretation
5. **Log transforms** -- `log_age`, `log_fee`, `log_quantity`, `log_photo_amt` to reduce right skew
6. **Age binning** -- Domain-informed bins (neonate to senior)
7. **Name features** -- `name_length`, `name_word_count` from valid names only
8. **Breed features** -- `is_mixed_breed`, `breed_count`, `breed1_frequency` (train-fitted)
9. **Color features** -- `color_count` + missing indicators
10. **Interaction features** -- `age_x_type`, `health_x_vaccinated` from EDA hypotheses
11. **Rescuer aggregates** -- `rescuer_pet_count`, `rescuer_mean_photo_amt`, `rescuer_mean_fee` (train-fitted, cold-start imputed)
12. **State frequency** -- `state_freq` from train distribution

**Artifacts produced:**
- `data/features/tabular/v1/train.parquet` (14,993 rows)
- `data/features/tabular/v1/test.parquet` (3,972 rows)
- `data/features/tabular/v1/schema.json`
- `reports/feature_engineering_tabular.json`

</div>

**Next:** Notebook `08_feature_extraction_text` (text and sentiment features) or `09_feature_extraction_images` (image features).